# 11.2 PPO 클리핑과 연속 제어 실습 — 노트북

[![Open In Colab: PPO](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SuminHan/book-ml/blob/main/notebooks/ml2/chapter11_2_ppo_clip_pendulum.ipynb)

책 11.2절의 세 가지를 실행으로 확인합니다:
1. **클리핑 목적함수의 모양** — \(L^{CLIP}\)의 한 샘플값을 손으로 계산하고 그래프로 그립니다.
2. **Pendulum-v1에서 PPO 전체 루프 학습** — Gaussian 정책 + GAE + 클리핑 + 엔트로피 보너스.
3. **Gaussian 정책의 그래디언트** — 책의 미분식을 수치 미분으로 검증합니다.

(학습은 CPU 기준 약 5분입니다.)

## 0. 설정: 한국어 폰트, 시드 고정, 이미지 저장 경로

In [1]:
import os, random, math
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib import font_manager
import torch
import torch.nn as nn
import torch.nn.functional as F
import gymnasium as gym

# 한국어 라벨을 폰트 'Noto Sans CJK KR'으로 (없으면 기본 폰트로 넘어가도 출력은 됨)
kr = [f.name for f in font_manager.fontManager.ttflist if "Noto Sans CJK KR" in f.name]
if kr:
    plt.rcParams["font.sans-serif"] = [kr[0]]
plt.rcParams["axes.unicode_minus"] = False

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

IMG = "/home/smhan/book-ml/kor/src/images"
if not os.path.isdir(IMG):   # Colab 등에서는 /tmp로 자동 대체
    IMG = "/tmp"
print(f"torch: {torch.__version__}, gymnasium: {gym.__version__}")
print(f"그림 저장 위치: {IMG}")

torch: 2.13.0+cpu, gymnasium: 1.3.0
그림 저장 위치: /home/smhan/book-ml/kor/src/images


## 1. 1부 — PPO 클리핑 목적함수: min이 만드는 "모서리"

11.1절에서 확률비 \(r_t = \pi_\theta / \pi_{\theta_{old}}\)를 소개했고,
11.2절은 이 식에 클리핑을 넣어 목적함수를 정했습니다:

\[L^{CLIP} = \mathbb{E}_t\left[\min\left(r_t A_t,\; \mathrm{clip}(r_t, 1-\epsilon, 1+\epsilon)\, A_t\right)\right]\]

아래 `ppo_clip_loss`는 이 기댓값의 **한 샘플분**을 계산합니다(책 11.2절 코드).
네 가지 경우를 직접 출력해서 확인하세요:

- `A=+1, r=0.5` → 범위를 안 벗어났으므로 원래 값 그대로
- `A=+1, r=1.5` → 1.2로 잘림 (추가 이득 0)
- `A=-1, r=1.5` → **원래 값(−1.5)이 채택**됨 (나쁜 것을 더 선호하는 방향엔 상한 없음)
- `A=-1, r=0.5` → −0.8로 고정됨 (나쁜 것을 덜 선호하는 방향의 추가 보상 0)

In [2]:
def ppo_clip_loss(ratio, advantage, epsilon=0.2):
    unclipped = ratio * advantage
    clipped = max(min(ratio, 1 + epsilon), 1 - epsilon) * advantage
    return min(unclipped, clipped)  # 목적함수(최대화 대상)의 한 샘플분

cases = [(0.5, 1.0), (1.5, 1.0), (1.5, -1.0), (0.5, -1.0)]
hdr = "%5s %5s %8s %8s %8s" % ("r_t", "A_t", "r*A", "clip*A", "L")
print(hdr)
for r, A in cases:
    un = r * A
    cl = max(min(r, 1.2), 0.8) * A
    print("%5.1f %5.1f %8.2f %8.2f %8.2f" % (r, A, un, cl, min(un, cl)))

  r_t   A_t      r*A   clip*A        L
  0.5   1.0     0.50     0.80     0.50
  1.5   1.0     1.50     1.20     1.20
  1.5  -1.0    -1.50    -1.20    -1.50
  0.5  -1.0    -0.50    -0.80    -0.80


두 패널 그래프로 이 "모서리"를 보겠습니다 — \(\epsilon=0.2\), \(r_t\)를 0~2.2까지:
- **왼쪽(A=+1, 좋은 행동)**: 검은 목적함수 곡선이 `r=1.2`에서 수평으로 고정이 됩니다.
- **오른쪽(A=-1, 나쁜 행동)**: 검은 곡선이 `r=0.8` **이하**에서 −0.8로 고정되고,
  `r=1.2` **이상**(나쁜 것을 더 선호하는 방향)에서는 원본 `r·A`를 따라 계속 내려갑니다.
  min이 "비관적(작은) 쪽"을 고르기 때문에 생기는 **비대칭**입니다.

In [3]:
def clip(r, e=0.2):
    return np.clip(r, 1 - e, 1 + e)

rs = np.linspace(0.0, 2.2, 400)
fig, axes = plt.subplots(1, 2, figsize=(9, 3.6))
for ax, A, t in zip(axes, [1.0, -1.0], ["어드밴티지 A = +1 (좋은 행동)", "어드밴티지 A = -1 (나쁜 행동)"]):
    ax.plot(rs, rs * A, "b-", lw=1.6, label="r·A (클립 없음)")
    ax.plot(rs, clip(rs) * A, "r--", lw=1.6, label="clip(r, 0.8, 1.2)·A")
    ax.plot(rs, np.minimum(rs * A, clip(rs) * A), "k-", lw=2.2, label="min(·, ·) — 실제 목적함수")
    ax.axvline(0.8, color="gray", ls=":", lw=1); ax.axvline(1.2, color="gray", ls=":", lw=1)
    ax.axhline(0, color="lightgray", lw=0.8)
    ax.set_xlabel("확률비 r"); ax.set_title(t)
    ax.legend(fontsize=8); ax.grid(alpha=0.3)
    if A > 0:
        ax.annotate("r > 1+ε: 추가 이득 0", xy=(1.2, 1.2), xytext=(1.35, 0.2),
                    fontsize=8, arrowprops=dict(arrowstyle="->", lw=0.8))
    else:
        ax.annotate('r < 1-ε: -0.8로 고정 ("더 억제" 보상 0)\nr > 1+ε: 원본 r·A를 따라 벌금 증가',
                    xy=(0.8, -0.8), xytext=(0.05, 0.35),
                    fontsize=8, arrowprops=dict(arrowstyle="->", lw=0.8))
fig.tight_layout()
fig.savefig(IMG + "/ch11_2_ppo_clip_loss.svg", bbox_inches="tight")
plt.show()
print("저장:", IMG + "/ch11_2_ppo_clip_loss.svg")

저장: /home/smhan/book-ml/kor/src/images/ch11_2_ppo_clip_loss.svg


## 2. 2부 — Pendulum-v1에서 PPO 전체 루프

Pendulum-v1: 3차원 상태 `[cosθ, sinθ, θ̇]`, 1차원 연속 행동(토크, [-2, +2] Nm),
리턴은 항상 음수(0에 가까울수록 좋음).

책 11.2절의 `GaussianPolicy`에 Critic 출력 \(V_\theta(s)\)을 더해
Actor-Critic 하나로 합친 구조(책의 전체 목적함수에서 세 항을 함께 갱신):

In [4]:
class ActorCritic(nn.Module):
    def __init__(self, state_dim, act_dim, act_high):
        super().__init__()
        self.trunk = nn.Sequential(nn.Linear(state_dim, 64), nn.Tanh(),
                                   nn.Linear(64, 64), nn.Tanh())
        self.mu = nn.Linear(64, act_dim)
        self.log_std = nn.Parameter(torch.zeros(act_dim) - 0.5)  # exp() 후 약 0.61
        self.v = nn.Linear(64, 1)
        self.act_high = act_high
    def forward(self, x):
        h = self.trunk(x)
        mu = torch.tanh(self.mu(h)) * self.act_high  # 행동 범위에 맞게 스케일
        std = torch.exp(self.log_std)
        return mu, std, self.v(h).squeeze(-1)

def compute_gae(rewards, values, last_value, gamma=0.99, lam=0.95, dones=None):
    """GAE: A_t = sum_k (gamma*lam)^k * delta_{t+k},  delta_t = r_t + gamma*V(s') - V(s)"""
    T = len(rewards)
    adv = [0.0] * T
    last_gae = 0.0
    for t in reversed(range(T)):
        next_v = values[t + 1] if t < T - 1 else last_value
        delta = rewards[t] + gamma * next_v - values[t]
        last_gae = delta + gamma * lam * last_gae
        adv[t] = last_gae
    rets = [a + v for a, v in zip(adv, values)]
    return adv, rets

### 2.1 학습 루프 (300 반복 × 400 스텝 = 12만 스텝)

책의 "전체 알고리즘 루프"를 그대로 구현합니다. 핵심 줄 3개를 짚자면:
- `ratio = exp(logp_new - logp_old)` — 11.1절의 \(r_t(\theta)\)
- `torch.min(surr1, surr2)` — 11.2절의 클리핑 (최대화가므로 손실에서는 `-min`)
- `entropy` — 탐험을 유지하는 보너스 (계수 0.01)

In [5]:
env = gym.make("Pendulum-v1")
ac = ActorCritic(env.observation_space.shape[0], env.action_space.shape[0],
                 env.action_space.high[0])
opt = torch.optim.Adam(ac.parameters(), lr=3e-4)

GAMMA, LAM, EPS, N_EPOCH, MB, N_ITER, STEPS, ENT_COEF, VAL_COEF = 0.99, 0.95, 0.2, 4, 64, 300, 400, 0.01, 0.5

episode_returns = []
cursor = 0  # 현재(분할된) 에피소드의 리턴 시작 인덱스
for it in range(N_ITER):
    state, _ = env.reset(seed=SEED + it)
    states, actions, logps, rewards, values = [], [], [], [], []
    for _ in range(STEPS):
        st = torch.tensor(state, dtype=torch.float32).unsqueeze(0)
        with torch.no_grad():
            mu, std, v = ac(st)
        dist = torch.distributions.Normal(mu, std)
        a = dist.sample()
        states.append(state); actions.append(a); logps.append(dist.log_prob(a).sum())
        next_state, r, term, trunc, _ = env.step(a.squeeze(0).numpy())
        rewards.append(r); values.append(v.item())
        if term or trunc:   # Pendulum은 truncation: 200 스�tep만 지나도 종료
            episode_returns.append(float(np.sum(rewards[cursor:])))
            cursor = len(rewards)
            state, _ = env.reset()
        else:
            state = next_state
    states_t = torch.tensor(np.asarray(states), dtype=torch.float32)
    actions_t = torch.tensor(np.asarray(actions), dtype=torch.float32)
    logps_t = torch.stack(logps)
    rewards_t = torch.tensor(rewards, dtype=torch.float32)
    values_t = torch.tensor(values, dtype=torch.float32)
    with torch.no_grad():
        last_v = ac(states_t[-1:])[2].item()
    adv, rets = compute_gae(rewards_t.tolist(), values_t.tolist(), last_v, GAMMA, LAM)
    adv_t = torch.tensor(adv, dtype=torch.float32)
    ret_t = torch.tensor(rets, dtype=torch.float32)
    adv_t = (adv_t - adv_t.mean()) / (adv_t.std() + 1e-8)  # 어드밴티지 정규화

    idx = torch.randperm(len(rewards))
    for _ in range(N_EPOCH):          # 같은 데이터를 4에폭 재사용 (이후 버림)
        for start in range(0, len(rewards), MB):
            b = idx[start:start + MB]
            mu, std, v = ac(states_t[b])
            dist = torch.distributions.Normal(mu, std)
            new_logps = dist.log_prob(actions_t[b]).sum(-1)
            ratio = torch.exp(new_logps - logps_t[b])       # <-- r_t(θ)
            surr1 = ratio * adv_t[b]
            surr2 = torch.clamp(ratio, 1 - EPS, 1 + EPS) * adv_t[b]
            pol_loss = -torch.min(surr1, surr2).mean()      # 최대화 -> 부호 반전
            val_loss = F.mse_loss(v, ret_t[b])
            entropy = dist.entropy().sum(-1).mean()
            loss = pol_loss + VAL_COEF * val_loss - ENT_COEF * entropy
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(ac.parameters(), 0.5)
            opt.step()
    if it % 25 == 0 or it == N_ITER - 1:
        print(f"iter {it:3d}: last10={np.mean(episode_returns[-10:]):.1f}")
print(f"\n총 에피소드: {len(episode_returns)}")

iter   0: last10=-1320.7


iter  25: last10=-758.7


iter  50: last10=-665.6


iter  75: last10=-751.1


iter 100: last10=-655.7


...

iter 200: last10=-604.9


iter 225: last10=-696.8


iter 250: last10=-724.0


iter 275: last10=-740.3


iter 299: last10=-741.4

총 에피소드: 600


### 2.2 결과: 정직한 학습 곡선

Pendulum 리턴은 항상 음수입니다(0에 가까울수록 좋음). 아래 그래프에서
개별 에피소드(연회색)의 큰 변동과, 25에피소드 이동평균(파랑)의 완만한
상승을 함께 보세요 — 책 11.2절에서 논의한 "12만 스텝으로는 아직
절반도 안 왔다"는 정직한 모습입니다(다른 seed에서는 숫자가 ±100~200
달라질 수 있음).

In [6]:
first10 = np.mean(episode_returns[:10])
last10 = np.mean(episode_returns[-10:])
print(f"처음 10 에피소드 평균 리턴:  {first10:.1f}")
print(f"마지막 10 에피소드 평균 리턴: {last10:.1f}")
print(f"최종 exp(log_std): {torch.exp(ac.log_std).item():.3f} (초기 {math.exp(-0.5):.3f})")

ep = np.arange(1, len(episode_returns) + 1)
sm = np.convolve(episode_returns, np.ones(25) / 25, mode="valid")
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(ep, episode_returns, color="lightgray", lw=0.7, alpha=0.8)
ax.plot(ep[24:], sm, color="tab:blue", lw=1.8)
ax.axhline(first10, color="gray", ls="--", lw=1)
ax.text(len(episode_returns) * 0.01, first10 + 80, f"처음 10 에피소드 평균 {first10:.0f}",
        fontsize=8, color="dimgray")
ax.axhline(last10, color="tab:red", ls="--", lw=1)
ax.text(len(episode_returns) * 0.55, last10 + 80, f"마지막 10 에피소드 평균 {last10:.0f}",
        fontsize=8, color="tab:red")
ax.set_xlabel("에피소드 번호"); ax.set_ylabel("리턴 (Pendulum, 음수일수록 좋음)")
ax.set_title("Pendulum-v1에서 PPO 학습 곡선 (시드 42)")
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(IMG + "/ch11_2_ppo_pendulum_curve.svg", bbox_inches="tight")
plt.show()
print("저장:", IMG + "/ch11_2_ppo_pendulum_curve.svg")

처음 10 에피소드 평균 리턴:  -805.3
마지막 10 에피소드 평균 리턴: -741.4
최종 exp(log_std): 1.325 (초기 0.607)
저장: /home/smhan/book-ml/kor/src/images/ch11_2_ppo_pendulum_curve.svg


## 3. 3부 — Gaussian 정책의 그래디언트 (책의 식 검증)

책 11.2절에서 유도한 식 — 정규분포의 로그가능도는
\(\log\pi(a|s) = -	frac{1}{2}((a-\mu)/\sigma)^2 - \log\sigma + 	ext{const}\)이며,
\(\mu\)와 \(\sigma\)로 각각 미분하면:

\[rac{\partial \log\pi}{\partial \mu} = rac{a-\mu}{\sigma^2},
\qquad
rac{\partial \log\pi}{\partial \sigma} = rac{(a-\mu)^2 - \sigma^2}{\sigma^3}\]

두 부분의 해석:
- **\(\mu\) 항**: 부호가 \(a-\mu\)로 정해짐 — "고른 행동 쪽으로 평균 이동"
  (10.2절 softmax one-vs-rest의 연속 버전).
- **\(\sigma\) 항**: 부호가 \((a-\mu)^2-\sigma^2\) — 행동이 평균에서 1σ 이상
  벗어나면(극단적 샘플) \(\sigma\)를 줄이고, 1σ 이내이면(평범한 샘플) 늘림.
  \(-\log\sigma\) 정규화 항에서 오는 \(-1/\sigma\) 성분이 이 부호 반전을 만든다.

아래에서 해석적 미분과 수치 미분(중심차분)이 일치하는지 확인합니다:

In [7]:
mu0, sigma0, a0 = 0.5, 0.3, 0.8

def logpi(a, mu, sigma):
    return -0.5 * ((a - mu) / sigma) ** 2 - math.log(sigma)

h = 1e-5
num_dmu = (logpi(a0, mu0 + h, sigma0) - logpi(a0, mu0 - h, sigma0)) / (2 * h)
num_dsigma = (logpi(a0, mu0, sigma0 + h) - logpi(a0, mu0, sigma0 - h)) / (2 * h)
ana_dmu = (a0 - mu0) / sigma0 ** 2
ana_dsigma = ((a0 - mu0) ** 2 - sigma0 ** 2) / sigma0 ** 3  # (a-mu)^2 - sigma^2 ; 여기 a0-mu0=sigma0 이므로 0

print(f"mu 항:    수치 {num_dmu:+.6f}  해석 {ana_dmu:+.6f}")
print(f"sigma 항: 수치 {num_dsigma:+.6f}  해석 {ana_dsigma:+.6f}  (a0-mu0=sigma0 -> 정확히 0)")
assert abs(num_dmu - ana_dmu) < 1e-4 and abs(num_dsigma - ana_dsigma) < 1e-4
print("\n해석적 미분 = 수치 미분  OK")

# 부호 확인: |a-mu| > sigma 이면 sigma 감소 방향, 안 되면 증가 방향
for a, label in [(mu0 + 2 * sigma0, "|a-mu|>sigma (극단)"), (mu0 + 0.5 * sigma0, "|a-mu|<sigma (평범)")]:
    d = ((a - mu0) ** 2 - sigma0 ** 2) / sigma0 ** 3
    print(f"a-mu = {a-mu0:+.1f} ({label}): dlogpi/dsigma = {d:+.4f} -> "
          + ("A>0이면 sigma 감소(탐험↓)" if d < 0 else "A>0이면 sigma 증가(탐험↑)"))

mu 항:    수치 +3.333333  해석 +3.333333
sigma 항: 수치 +0.000000  해석 +0.000000  (a0-mu0=sigma0 -> 정확히 0)

해석적 미분 = 수치 미분  OK
a-mu = +0.6 (|a-mu|>sigma (극단)): dlogpi/dsigma = +10.0000 -> A>0이면 sigma 증가(탐험↑)
a-mu = +0.2 (|a-mu|<sigma (평범)): dlogpi/dsigma = -2.5000 -> A>0이면 sigma 감소(탐험↓)


## 정리

- **클리핑**은 \(r_t\)의 *개선 방향* 이동(좋은 것을 더, 나쁜 것을 덜)에만
  상한을 걸고, *악화 방향* 이동에는 걸지 않는다 — min의 "비관적 선택"이
  만들어내는 비대칭.
- **PPO 루프** = 데이터 수집 → GAE → (에폭×미니배치) 클리핑 갱신 → 데이터
  버림. 재현 버퍼(Ch9.4)와 정반대로 데이터를 몇 에폭만 쓰고 버린다.
- **연속 제어**의 느린 학습은 구현의 결함이 아니라 행동공간의 탐색 체적
  문제 — Ch13~14 로봇 시뮬레이션은 수백만 스텝을 계획한다.